In [84]:
import pandas as pd
import numpy as np

In [85]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_ibm_runtime.fake_provider import FakeAthensV2

from qiskit.circuit.library import n_local
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [86]:
atom_string = f"H 0 0 0; H 0 0 {0.5}"

driver = PySCFDriver(
    atom=atom_string,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

es_problem = driver.run()

# dapatkan qubit hammiltonian
second_q_op = es_problem.hamiltonian.second_q_op()

mapper = JordanWignerMapper()
hamiltonian = mapper.map(second_q_op)

ansatz = n_local(4,
        rotation_blocks=['ry'],
        entanglement_blocks=['cx'],
        entanglement="linear",
        reps=2,
        insert_barriers=True)

theta_arr = [np.random.uniform(-5, 5) for param in ansatz.parameters]


In [87]:
hamiltonian

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.67852307+0.j,  0.21393531+0.j, -0.36914432+0.j,  0.21393531+0.j,
 -0.36914432+0.j,  0.1345924 +0.j,  0.17992651+0.j,  0.17680996+0.j,
  0.04221756+0.j,  0.04221756+0.j,  0.04221756+0.j,  0.04221756+0.j,
  0.17680996+0.j,  0.18620984+0.j,  0.1345924 +0.j])

In [88]:
ansatz.draw()

┌──────────┐ ░                 ░ ┌──────────┐ ░                 ░ »
q_0: ┤ Ry(θ[0]) ├─░───■─────────────░─┤ Ry(θ[4]) ├─░───■─────────────░─»
     ├──────────┤ ░ ┌─┴─┐           ░ ├──────────┤ ░ ┌─┴─┐           ░ »
q_1: ┤ Ry(θ[1]) ├─░─┤ X ├──■────────░─┤ Ry(θ[5]) ├─░─┤ X ├──■────────░─»
     ├──────────┤ ░ └───┘┌─┴─┐      ░ ├──────────┤ ░ └───┘┌─┴─┐      ░ »
q_2: ┤ Ry(θ[2]) ├─░──────┤ X ├──■───░─┤ Ry(θ[6]) ├─░──────┤ X ├──■───░─»
     ├──────────┤ ░      └───┘┌─┴─┐ ░ ├──────────┤ ░      └───┘┌─┴─┐ ░ »
q_3: ┤ Ry(θ[3]) ├─░───────────┤ X ├─░─┤ Ry(θ[7]) ├─░───────────┤ X ├─░─»
     └──────────┘ ░           └───┘ ░ └──────────┘ ░           └───┘ ░ »
«      ┌──────────┐
«q_0: ─┤ Ry(θ[8]) ├
«      ├──────────┤
«q_1: ─┤ Ry(θ[9]) ├
«     ┌┴──────────┤
«q_2: ┤ Ry(θ[10]) ├
«     ├───────────┤
«q_3: ┤ Ry(θ[11]) ├
«     └───────────┘

In [89]:
from qiskit_ibm_runtime import EstimatorV2 as ZNE_estimator
from qiskit_aer.noise import NoiseModel

zne_estimator = ZNE_estimator(mode=FakeAthensV2())
zne_estimator.options.resilience.zne_mitigation = True
zne_estimator.options.resilience.zne.noise_factors = (1, 3, 5)
zne_estimator.options.resilience.zne.extrapolator = "exponential"

# Get the dictionary of transpiled operation counts
pm = generate_preset_pass_manager(
    backend=FakeAthensV2(),
    optimization_level=0,
    seed_transpiler=1,
    initial_layout=[0, 1, 2, 3]   
) 

ansatz = pm.run(ansatz)

In [94]:
hamiltonian.num_qubits

5

In [91]:
ansatz.layout

TranspileLayout(initial_layout=Layout({
0: <Qubit register=(4, "q"), index=0>,
1: <Qubit register=(4, "q"), index=1>,
2: <Qubit register=(4, "q"), index=2>,
3: <Qubit register=(4, "q"), index=3>,
4: <Qubit register=(1, "ancilla"), index=0>
}), input_qubit_mapping={<Qubit register=(4, "q"), index=0>: 0, <Qubit register=(4, "q"), index=1>: 1, <Qubit register=(4, "q"), index=2>: 2, <Qubit register=(4, "q"), index=3>: 3, <Qubit register=(1, "ancilla"), index=0>: 4}, final_layout=None, _input_qubit_count=4, _output_qubit_list=[<Qubit register=(5, "q"), index=0>, <Qubit register=(5, "q"), index=1>, <Qubit register=(5, "q"), index=2>, <Qubit register=(5, "q"), index=3>, <Qubit register=(5, "q"), index=4>])

In [92]:
hamiltonian = hamiltonian.apply_layout(ansatz.layout)

In [95]:

pub = (ansatz, hamiltonian, theta_arr)
job = zne_estimator.run([pub])
result = job.result()
energy = float(result[0].data.evs)

type(energy)

/home/fadhil/ml_qemV2/.venv/lib/python3.13/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:273: UserWarning: Options {'resilience': {'zne_mitigation': True, 'zne': {'noise_factors': (1.0, 3.0, 5.0), 'extrapolator': 'exponential'}}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")


float

In [1]:
import covalent as ct

# Construct manageable tasks out of functions
# by adding the @ct.electron decorator
@ct.electron
def add(x, y):
   return x + y

@ct.electron
def multiply(x, y):
   return x*y

# Note that electrons can be shipped to variety of compute
# backends using executors, for example, "local" computer.
# See below for other common executors.
@ct.electron(executor="local")
def divide(x, y):
   return x/y

# Construct the workflow by stitching together
# the electrons defined earlier in a function with
# the @ct.lattice decorator
@ct.lattice
def workflow(x, y):
   r1 = add(x, y)
   r2 = [multiply(r1, y) for _ in range(4)]
   r3 = [divide(x, value) for value in r2]
   return r3

# Dispatch the workflow
dispatch_id = ct.dispatch(workflow)(1, 2)
result = ct.get_result(dispatch_id)
print(result)


Lattice Result
status: RUNNING
result: None
input args: 1, 2
input kwargs: 
error: None

start_time: 2025-12-09 11:45:55.870719
end_time: None

results_dir: /home/fadhil/.cache/covalent/results/5c3aa912-28ae-4b8b-81f0-14756e866204
dispatch_id: 5c3aa912-28ae-4b8b-81f0-14756e866204

Node Outputs
------------
add(0): None
:parameter:1(1): 1
:parameter:2(2): 2
multiply(3): None
:parameter:2(4): 2
multiply(5): None
:parameter:2(6): 2
multiply(7): None
:parameter:2(8): 2
multiply(9): None
:parameter:2(10): 2
divide(11): None
:parameter:1(12): 1
divide(13): None
:parameter:1(14): 1
divide(15): None
:parameter:1(16): 1
divide(17): None
:parameter:1(18): 1
:postprocess:reconstruct(19): None
:electron_list:(20): None



In [ ]:
import covalent as ct

# Construct manageable tasks out of functions
# by adding the @ct.electron decorator
@ct.electron
def add(x, y):
   return x + y

@ct.electron
def multiply(x, y):
   return x*y

# Note that electrons can be shipped to variety of compute
# backends using executors, for example, "local" computer.
# See below for other common executors.
@ct.electron(executor="local")
def divide(x, y):
   return x/y

# Construct the workflow by stitching together
# the electrons defined earlier in a function with
# the @ct.lattice decorator
@ct.lattice
def workflow(x, y):
   r1 = add(x, y)
   r2 = [multiply(r1, y) for _ in range(4)]
   r3 = [divide(x, value) for value in r2]
   return r3

# Dispatch the workflow
dispatch_id = ct.dispatch(workflow)(1, 2)
result = ct.get_result(dispatch_id)
print(result)


Lattice Result
status: RUNNING
result: None
input args: 1, 2
input kwargs: 
error: None

start_time: 2025-12-09 11:45:55.870719
end_time: None

results_dir: /home/fadhil/.cache/covalent/results/5c3aa912-28ae-4b8b-81f0-14756e866204
dispatch_id: 5c3aa912-28ae-4b8b-81f0-14756e866204

Node Outputs
------------
add(0): None
:parameter:1(1): 1
:parameter:2(2): 2
multiply(3): None
:parameter:2(4): 2
multiply(5): None
:parameter:2(6): 2
multiply(7): None
:parameter:2(8): 2
multiply(9): None
:parameter:2(10): 2
divide(11): None
:parameter:1(12): 1
divide(13): None
:parameter:1(14): 1
divide(15): None
:parameter:1(16): 1
divide(17): None
:parameter:1(18): 1
:postprocess:reconstruct(19): None
:electron_list:(20): None

